## Classic speeches 
In this project, I try to assess the complexity level of central bank speeches, focusing on three central banks : the ECB, the Fed and the BoE. To do so, I rely on a dataset of speeches from these three institutions. This dataset comes from this article : Campiglio, E., Deyris, J., Romelli, D. and Scalisi, G., 2025. Warning words in a warming world: Central bank communication and climate change. European Economic Review [Open Access], Vol 178.

In [1]:
import sys
sys.path.append('/home/onyxia/work/nlp_central_banks/lyna_work')

from module import *

In [2]:
# Database of Campiglio, Deyris, Romelli and Scalisi (saved in the SSP Cloud, too heavy for github) :
MY_BUCKET = "lelkamel"
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"}
)
files_url = f"{MY_BUCKET}/"
fs.get(f"{MY_BUCKET}/","", recursive=True)

#Import
specialized_speeches = pd.read_csv("CBS_dataset_v1.0.csv")

#### Checking the data and selecting the subsample of interest
In this part, I start by selecting the sample that interests me, that is, speeches from the BoE, the Fed and the ECB starting 2001.

In [3]:
#I start my analysis in 2001
specialized_speeches["Date"] = pd.to_datetime(specialized_speeches["Date"])
specialized_speeches["Year"] = specialized_speeches["Date"].dt.year
specialized_speeches = specialized_speeches[specialized_speeches["Year"] > 2000]

# I only select the Fed, the BoE and the ECB
countries_selected = [
    "USA",
    "GBR",
    "ECB"
]
specialized_speeches = specialized_speeches[
    specialized_speeches["Country"].isin(countries_selected)
]

selected_banks = [
    "Bank of England",
    "Board of Governors of the Federal Reserve",
    "European Central Bank"
]
specialized_speeches_filtered = specialized_speeches[specialized_speeches["CentralBank"].isin(selected_banks)].copy()

In [4]:
df_topic = specialized_speeches_filtered.copy()
df_topic.head(3)

,URL,PDF,Title,Subtitle,Date,Authorname,Role,Gender,CentralBank,Country,text,text_original,Filename,Language,Source,Year
3049,https://www.ecb.europa.eu/press/key/date/2001/...,NaN,Lessons from the European experience with exch...,"Speech delivered by Christian Noyer, Vice...",2001-01-14,Christian Noyer,Deputy Governor,Male,European Central Bank,ECB,Lessons from the European experience with exch...,NaN,ecb_010114.en,English,BIS,2001
3050,https://www.ecb.europa.eu/press/key/date/2001/...,NaN,Some ECB views on the accession process,"Speech delivered by Christian Noyer, Vice-Pre...",2001-01-17,Christian Noyer,Deputy Governor,Male,European Central Bank,ECB,Some ECB views on the accession process Speech...,NaN,ecb_010117.en,English,BIS,2001
3051,https://www.ecb.europa.eu/press/key/date/2001/...,NaN,L'euro et l'volution du paysage financier en E...,"Allocution prononce par Christian Noyer, V...",2001-01-19,Christian Noyer,Deputy Governor,Male,European Central Bank,ECB,The euro and the changing financial landscape ...,NaN,ecb_010119.en,English,CB websites,2001


The dataset contains the following columns : 
- `URL` : the URL address of the speech, if collected from a website
- `PDF` : the address of the PDF file
- `Title` : the title of the speech
- `Subtitle` : the subtitle of the speech, when available
- `Date` : the date of the speech
- `Authorname` : the full name of the speaker
- `Role` : the institutional role of the speaker, coded as `Governor`, 
  `Deputy Governor`, `Board member`, or `Senior management`
- `Gender` : the gender of the speaker, coded as `Male` or `Female`
- `CentralBank` : the institution of the speaker
- `Country` : the country of the central bank
- `text` : the full text of the speech, translated to English when necessary
- `text_original` : the original text of the speech, when it was not in English
- `Filename` : the filename of the source document
- `Language` : the original language of the speech
- `Source` : the data source, coded as `BIS`, `CB websites`, or `Archives`
- `Year` : the year of the speech

#### Checks to prepare the data for NLP tasks
In this part, I make sure the text is well formated. I look and correct serveral artifacts. 

In [5]:
# --- First, I do some verifications to see how speeches are formatted ---
print(df_topic['Language'].value_counts())
mask = df_topic['text_original'].notna()
print(df_topic[mask][['text', 'text_original']].head(2)) 

Language
English    5219
German       76
Spanish      28
French       12
Italian       5
Name: count, dtype: int64
                                                   text  \
3063  Inflation, a key variable for the ECB's moneta...   
3065  Europe and the new economy Eugenio Domingo Sol...   

                                          text_original  
3063  La inflacin, una variable clave para la poltic...  
3065  Europa y la nueva economa Eugenio Domingo Sola...  


 As expected, some speeches are not in English. For example, there are speeches in German, French, Italian. These must be ECB speeches. However, the good news is, that at said in Campiglio, all texts were translated in English !

In [6]:
#By analysing a speech which was a PDF on an LLM, I see that it still has PDF page numbers in the “1/2001” format 
#  how widely can we find some in the dataset ? 
print(df_topic.groupby(['Source', df_topic['PDF'].notna()])['text'].apply(
    lambda x: x.str.contains(r'\d{1,2}/\d{4}\s+\d+', regex=True, na=False).mean()
).round(2))

Source       PDF  
Archives     True     0.00
BIS          False    0.00
             True     0.39
CB websites  False    0.00
             True     0.01
Name: text, dtype: float64


I also look for some artifacts in PDFs, for example here it seems that the page numbers in format "1/2001" are specific to BIS documents. I generalise this reasoning and use the metadata to look for common pattern in order to better clean it in a function that will rely on a dictionnary.

In [7]:
# With the help of an LLM (Claude ai), I write this code to check more systematically for other recurrent patterns
patterns = {
    'footer_ECB':     r'CONTACT European Central Bank',
    'footer_BOE':     r'Bank of England.*?©',
    'footer_FED':     r'Federal Reserve.*?Board',
    'page_number':    r'\b\d{1,2}/\d{4}\b',
    'encoding_artefact': r'[A-Za-z]\d[A-Za-z]\d',  # "Rf6ntgen"
    'references':     r'\b(References|Bibliography)\b',
    'tables':         r'(Table \d|Figure \d)',
    'footnotes':      r'^\d{1,2}\.',          # lignes commençant par "1."
    'urls':           r'https?://\S+',
    'brackets_ref':   r'\(\w+,\s*\d{4}\)',    # "(Nickell, 1997)"
    'squarebrackets': r'\[\d+\]',             # "[1]"
}

for name, pat in patterns.items():
    count = df_topic['text'].str.contains(pat, regex=True, na=False).sum()
    pct = count / len(df_topic) * 100
    print(f"{name:25s} : {count:4d} discours ({pct:.1f}%)")


footer_ECB                : 2097 discours (39.3%)
footer_BOE                :    0 discours (0.0%)
footer_FED                : 1602 discours (30.0%)
page_number               :  860 discours (16.1%)
encoding_artefact         :   65 discours (1.2%)


/tmp/ipykernel_23769/3848981163.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = df_topic['text'].str.contains(pat, regex=True, na=False).sum()


references                :  747 discours (14.0%)
tables                    :  512 discours (9.6%)
footnotes                 :    0 discours (0.0%)
urls                      :    1 discours (0.0%)
brackets_ref              :  337 discours (6.3%)
squarebrackets            : 1537 discours (28.8%)


Note that some speech have a reference section ! We will have to remove them to for the topic modelling to work well. 

In [14]:
# ── References section ─────────────────────────────────────
print("\n=== Speeches with References section ===")
has_refs = df_topic['text'].str.contains(r'\b(References|Bibliography)\b', regex=True, na=False)
sample_ref = df_topic[has_refs]['text'].iloc[0]
match = re.search(r'\b(References|Bibliography)\b', sample_ref)
print(repr(sample_ref[max(0, match.start()-100):match.end()+300]))


=== Speeches with References section ===


/tmp/ipykernel_23769/2963717491.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_refs = df_topic['text'].str.contains(r'\b(References|Bibliography)\b', regex=True, na=False)


'in this "revolution" by ensuring the enabling factor of price stability. Annex: [pdf, 94kb] List of References Barber, W. J. (ed.) (1977), "The Works of I. Fisher", Pickering and Chatto. Bassanini, A., S. Scarpetta and I. Visco (2000), "Knowledge, technology and economic growth: recent evidence from OECD countries", mimeo, OECD, Paris , May. Bordo, M., B. Eichengreen and I. Douglas (1999), "Is Globalization'


Furthermore, in the BoE speeches, it seems that we also have some working papers.

In [15]:
# ── Illustrate the Nickell case ────────────────────────────
nickell = df_topic[df_topic['Authorname'].str.contains('Nickell', na=False)].iloc[0]

print("=== Example of an academic-style BoE speech ===\n")
print(f"Author  : {nickell['Authorname']}")
print(f"Role    : {nickell['Role']}")
print(f"Year    : {nickell['Year']}")
print(f"Words   : {nickell['n_words']}")
print(f"Source  : {nickell['Source']}")
print()

# Count tables, references, citations
text = nickell['text']
n_tables    = len(re.findall(r'Table \d+', text))
n_figures   = len(re.findall(r'Figure \d+', text))
n_citations = len(re.findall(r'\(\w+,\s*\d{4}\)', text))
has_refs    = bool(re.search(r'\bReferences\b', text))

print(f"Tables found       : {n_tables}")
print(f"Figures found      : {n_figures}")
print(f"In-text citations  : {n_citations}")
print(f"References section : {has_refs}")
print()

# Compare average length by role and bank
print("=== Average speech length by role and central bank ===\n")
print(
    df_topic.groupby(['CentralBank', 'Role'])['n_words']
    .agg(['count', 'mean', 'max'])
    .round(0)
    .sort_values('mean', ascending=False)
    .to_string()
)

print()

# Show the difference after filtering
print("=== Average length : full sample vs. filtered sample ===\n")
comparison = pd.DataFrame({
    'Full sample' : df_topic.groupby('CentralBank')['n_words'].mean().round(0),
    'Gov. & Deputy Gov.' : df_filtered.groupby('CentralBank')['n_words'].mean().round(0),
}).sort_values('Full sample', ascending=False)

print(comparison.to_string())
print()
print("→ Restricting to Governors and Deputy Governors reduces average speech")
print("  length and ensures comparability across the three institutions.")

=== Example of an academic-style BoE speech ===

Author  : Stephen Nickell
Role    : Board member
Year    : 2002


KeyError: 'n_words'

Thanks to this initial exploratory work, I now have enough data to write a database cleanup function tailored to the different text formats. Before moving on to this step, I also examine the distribution of text lengths by central bank and by year to ensure there are no anomalies. 

In [ ]:
# ------ Now we check for the distribution of lenght ---------
df_topic['n_words'] = df_topic['text'].str.split().str.len()

print(df_topic.groupby(['CentralBank', 'Source'])['n_words'].describe().round(0))


# --- Look at the origin of very short and very long texts --------
print("\n--- Textes très courts (< 500 mots) ---")
print(df_topic[df_topic['n_words'] < 500][['CentralBank','Source','Year','n_words']].head(10))

print("\n--- Textes très longs (> 15000 mots) ---")
print(df_topic[df_topic['n_words'] > 15000][['CentralBank','Source','Year','n_words']].head(10))

                                                        count    mean     std  \
CentralBank                               Source                                
Bank of England                           BIS           716.0  4488.0  2578.0   
                                          CB websites   531.0  4873.0  3282.0   
Board of Governors of the Federal Reserve Archives       26.0  3496.0  2609.0   
                                          BIS          1519.0  3240.0  1597.0   
                                          CB websites   135.0  2951.0  1766.0   
European Central Bank                     BIS          1703.0  2948.0  1669.0   
                                          CB websites   710.0  3259.0  1792.0   

                                                          min     25%     50%  \
CentralBank                               Source                                
Bank of England                           BIS           528.0  2763.0  3857.0   
                           

In [ ]:
# Are short texts usable ?
short = df_topic[df_topic['n_words'] < 500]
print(short['text'].iloc[0][:500])

# Are BoE long texts only academic papers ?
long_boe = df_topic[df_topic['n_words'] > 15000].iloc[0]
print(long_boe['text'][:300])
print("...")
print(long_boe['text'][-300:])

I make sure there are no errors in the data. Indeed each country is associated with its institution of reference : Brittain is the BoE, the European union is the EU and the USA, the state institutions.

### Some descriptive statistics

Do some instutions speak more than others ? Who speaks ?

In [ ]:
df = specialized_speeches_filtered.copy()
country_year = (
    df.groupby(['Year', 'Country'])
      .size()
      .reset_index(name='Count')
)
# Country evolution
plt.figure(figsize=(14, 7))
sns.lineplot(data=country_year, x='Year', y='Count', hue='Country', marker='o', linewidth=2)
plt.grid(True, alpha=0.3)
plt.title('Evolution of Speeches by Country')
plt.xlabel('Year')
plt.ylabel('Number of Speeches')
plt.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


We consider several metrics that aim to capture the **cognitive complexity** of texts—such as sentence length, document length, and grammatical structure.

However, as highlighted in a *June 2025 International Monetary Fund working paper*, these indicators, while informative, **do not fully capture the semantic content of communication**, including the underlying economic rationale, policy intent, or strategic framing. To gain deeper insights, a more advanced NLP analysis is therefore required.

At this stage, we focus on the following metrics:

1. **Average word count by institution**  
   Measures the typical length of documents and provides a proxy for the overall verbosity of communication.

2. **Average sentence length**  
   Captures syntactic complexity at the sentence level, with longer sentences generally indicating higher processing effort.

3. **Flesch–Kincaid score**  
   Evaluates readability and linguistic difficulty based on sentence length and word complexity.

4. **Dependency depth by institution**  
   Reflects grammatical complexity by measuring the depth of syntactic dependency trees.

These metrics provide a useful first layer of analysis for comparing communication styles across institutions, but remain limited to surface-level textual features.

In [ ]:
import numpy as np
from textstat import flesch_reading_ease
import seaborn as sns
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
nltk.download('punkt_tab')

In [ ]:
#Varable to study speech complexity (6 minutes)
df['word_count'] = df['text'].apply(lambda x: len(word_tokenize(x))) 
df['char_count'] = df['text'].apply(len)
df['sentence_count'] = df['text'].apply(lambda x: len(sent_tokenize(x)))
df['avg_word_length'] = df['text'].apply(lambda x: np.mean([len(word) for word in word_tokenize(x)]) if len(word_tokenize(x)) > 0 else 0)
df['avg_sentence_length'] = df['word_count'] / df['sentence_count']
df['flesch_score'] = df['text'].apply(lambda x: flesch_reading_ease(x))

In [ ]:
figures_path = "/home/onyxia/work/nlp_central_banks/lyna_work/figures"
os.makedirs(figures_path, exist_ok=True)

countries = ['ECB', 'GBR', 'USA']
labels = {'ECB': 'ECB', 'GBR': 'BoE', 'USA': 'FED'}

df_plot = df[df['Country'].isin(countries)].copy()
df_plot['Institution'] = df_plot['Country'].map(labels)
df_plot['Year'] = df_plot['Year'].astype(int)

yearly = (
    df_plot
    .groupby(['Institution', 'Year'])
    .agg({
        'word_count': 'mean',
        'avg_sentence_length': 'mean',
        'flesch_score': 'mean',
        'dependency_depth': 'mean'
    })
    .reset_index()
)
metrics_1 = ['word_count', 'avg_sentence_length']
titles_1 = ['Word Count', 'Average Sentence Length']

fig, axs = plt.subplots(2, 2, figsize=(16, 10))

for i, (metric, title) in enumerate(zip(metrics_1, titles_1)):
    sns.boxplot(
        data=df_plot,
        x='Institution',
        y=metric,
        ax=axs[0, i]
    )
    axs[0, i].set_title(f'{title} by Institution')
    axs[0, i].grid(True, alpha=0.3)

    for inst in ['ECB', 'BoE', 'FED']:
        inst_data = yearly[yearly['Institution'] == inst]
        axs[1, i].plot(
            inst_data['Year'],
            inst_data[metric],
            marker='o',
            linewidth=2,
            label=inst
        )

    axs[1, i].set_title(f'{title} Over Time')
    axs[1, i].set_xlabel('Year')
    axs[1, i].set_ylabel(title)
    axs[1, i].grid(True, alpha=0.3)
    axs[1, i].legend()

plt.tight_layout()
plt.savefig(
    f"{figures_path}/speech_complexity_word_sentence_length.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()

In [ ]:
from tqdm import tqdm
import pydantic
import spacy
import thinc

print("pydantic:", pydantic.__version__)
print("spacy:", spacy.__version__)
print("thinc:", thinc.__version__)

nlp = spacy.load("en_core_web_sm")
print("OK")


# -----------------------------
# 3. Function to compute dependency depth of one sentence
# -----------------------------
def sentence_dependency_depth(sent):
    """
    Depth = longest path from sentence root to any terminal node (leaf).
    We count nodes on the path, so a root->child->grandchild path has depth 3.
    """
    root = sent.root

    def depth_from_token(token):
        children = list(token.children)
        if not children:
            return 1
        return 1 + max(depth_from_token(child) for child in children)

    return depth_from_token(root)

# -----------------------------
# 4. Function to compute speech-level dependency depth
# -----------------------------
def speech_dependency_depth(text):
    """
    Compute average sentence dependency depth for one speech.
    You could also use max(sentence depths) instead of mean if preferred.
    """
    doc = nlp(text)

    sentence_depths = []
    for sent in doc.sents:
        # skip extremely short / empty-like sentences
        sent_tokens = [tok for tok in sent if not tok.is_space]
        if len(sent_tokens) == 0:
            continue
        sentence_depths.append(sentence_dependency_depth(sent))

    if len(sentence_depths) == 0:
        return None

    return sum(sentence_depths) / len(sentence_depths)

#df['dependency_depth'] = df['text'].apply(speech_dependency_depth)
# Drop rows where parsing failed
#df = df.dropna(subset=['dependency_depth'])


output_file = "df_with_depth.csv"

# Reprise si fichier existe
if os.path.exists(output_file):
    df_saved = pd.read_csv(output_file)
    df['dependency_depth'] = df_saved['dependency_depth']
else:
    df['dependency_depth'] = None

# Boucle avec barre de progression
for i in tqdm(range(len(df)), desc="Processing speeches"):
    
    if pd.notnull(df.loc[i, 'dependency_depth']):
        continue  # skip déjà fait

    text = df.loc[i, 'text']
    
    try:
        depth = speech_dependency_depth(text)
    except Exception as e:
        depth = None
        print(f"\nError at index {i}: {e}")  # \n pour ne pas casser la barre

    df.loc[i, 'dependency_depth'] = depth

    # 💾 Sauvegarde tous les 10
    if i % 10 == 0:
        df.to_csv(output_file, index=False)

# Sauvegarde finale
df.to_csv(output_file, index=False)
print("Final save complete")

So the speeches seem to be shorter across time (except for the UK). Now, how readable are they ? I start to study leaxical readability metrics (such as the Flesch-Kincaid scores which are widely used over the literature). I also consider a grammatical metric, using dependency parsing (note that I do not use the most developed models, as we are only in a preprocessing step).

The Flesch-Kincaid Ease Score is calculated as:
$$
\text{Score } = 206.835-1.015(\frac{\text{ Total Words }}{\text{ Total Sentences }})-84.6(\frac{\text{ Total Syllables }}{\text{ Total Words }})
$$
The score ranges from 0 to 100, with higher values indicating better readability. In English-language texts, scores above 60 correspond to an 8th-grade reading level, while values between 30 and 50 suggest college-level complexity. Scores below 30 indicate highly technical or specialized content.

In [ ]:
df = pd.read_csv('/home/onyxia/work/nlp_central_banks/lyna_work/speeches_cleaned.csv')

In [ ]:
metrics_2 = ['flesch_score', 'dependency_depth']
titles_2 = ['Flesch/Kincaid Score', 'Dependency Depth']

fig, axs = plt.subplots(2, 2, figsize=(16, 10))

for i, (metric, title) in enumerate(zip(metrics_2, titles_2)):
    sns.boxplot(
        data=df_plot,
        x='Institution',
        y=metric,
        ax=axs[0, i]
    )
    axs[0, i].set_title(f'{title} by Institution')
    axs[0, i].grid(True, alpha=0.3)

    for inst in ['ECB', 'BoE', 'FED']:
        inst_data = yearly[yearly['Institution'] == inst]
        axs[1, i].plot(
            inst_data['Year'],
            inst_data[metric],
            marker='o',
            linewidth=2,
            label=inst
        )

    axs[1, i].set_title(f'{title} Over Time')
    axs[1, i].set_xlabel('Year')
    axs[1, i].set_ylabel(title)
    axs[1, i].grid(True, alpha=0.3)
    axs[1, i].legend()

plt.tight_layout()
plt.savefig(
    f"{figures_path}/speech_complexity_flesch_dependency.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()

#### Data Preprocessing

Now, I select the speeches that look the most alike. For example, we can see that state central bank speeches in the US are on average shorter than federal speeches or than that of the ECB and the BOE. Thus, for the US, we will only focus on the Fed speeches.

In [ ]:
pd.set_option('display.max_colwidth', None)
print(df_topic.loc[0, "text"])

In [ ]:
# Application
df_topic['text_clean'] = df_topic.apply(clean_text, axis=1)
df_topic['n_words_clean'] = df_topic['text_clean'].str.split().str.len()

diff = df_topic['n_words'] - df_topic['n_words_clean']

print(diff.describe().round(1))
print(f"\nDiscours avec > 200 mots supprimés  : {(diff > 200).sum()}")
print(f"Discours avec > 500 mots supprimés  : {(diff > 500).sum()}")
print(f"Discours avec > 1000 mots supprimés : {(diff > 1000).sum()}")
print(f"Discours avec > 2000 mots supprimés : {(diff > 2000).sum()}")

# Vérification sur les cas problématiques
print("\n--- Cas cibles ---")
for idx in [4871, 2532, 2512, 4167, 4924]:
    orig  = df_topic.loc[idx, 'text']
    clean = df_topic.loc[idx, 'text_clean']
    print(f"Index {idx} | {df_topic.loc[idx, 'Year']} | {len(orig.split())} -> {len(clean.split())} mots")

In [ ]:
worst = df_topic.assign(diff=df_topic['n_words'] - df_topic['n_words_clean'])
worst_12 = worst.nlargest(12, 'diff')[['CentralBank', 'Source', 'Year', 'n_words', 'n_words_clean', 'diff']]
print(worst_12)

# Pour chacun, montrer le contexte de coupure
for idx in worst_12.index:
    orig  = df_topic.loc[idx, 'text']
    clean = df_topic.loc[idx, 'text_clean']
    cut   = len(clean)
    print(f"\nIndex {idx} | {df_topic.loc[idx, 'CentralBank'][:3]} | {df_topic.loc[idx, 'Year']}")
    print(repr(orig[max(0, cut-100):cut+100]))

Récapitulatif du preprocessing accompli jusqu'à présent : 
1) Supression propre des sections Rferences/Bibliography (papers académiques de la BoE ou de la Fed)
2) Footer ECB supprimés
3) Notes de bas de page [1] supprimés
4) Numéros de page PDF supprimés
5) Artefacts d'encoding corrigés
6) Apostrophes Fed corrigées

Now, we have to select only the texts that are alike. So that Bertopic performs well.

In [ ]:
# Distribution des rôles
print(df_topic['Role'].value_counts())

# Longueur moyenne par rôle
print(df_topic.groupby('Role')['n_words'].agg(['count', 'mean', 'std']).round(0).sort_values('count', ascending=False))

# Croisement rôle x banque
print(pd.crosstab(df_topic['Role'], df_topic['CentralBank']))

In [ ]:
 Qui sont les "Board members" de la Fed ?
print("=== FED Board members - exemples ===")
print(df_topic[
    (df_topic['CentralBank'] == 'Board of Governors of the Federal Reserve') &
    (df_topic['Role'] == 'Board member')
]['Authorname'].value_counts().head(10))

# Qui sont les "Governors" par banque ?
print("\n=== Governors par banque ===")
print(df_topic[df_topic['Role'] == 'Governor'].groupby(
    ['CentralBank', 'Authorname']
)['Role'].count().reset_index().sort_values(['CentralBank', 'Role'], ascending=[True, False]))

# Les Deputy Governors - quelle banque ?
print("\n=== Deputy Governors par banque ===")
print(pd.crosstab(df_topic[df_topic['Role'] == 'Deputy Governor']['CentralBank'],
                  df_topic[df_topic['Role'] == 'Deputy Governor']['Role']))

# Longueur par rôle ET banque
print("\n=== Longueur moyenne par rôle x banque ===")
print(df_topic.groupby(['CentralBank', 'Role'])['n_words'].agg(['count','mean']).round(0))

In [ ]:
df_filtered = df_topic[df_topic['Role'].isin(['Governor', 'Deputy Governor'])].copy()

print(f"Corpus original  : {len(df_topic)} discours")
print(f"Corpus filtré    : {len(df_filtered)} discours")
print(f"\nPar banque :")
print(df_filtered.groupby(['CentralBank', 'Role'])['Authorname'].count())
print(f"\nLongueur moyenne après filtre :")
print(df_filtered.groupby('CentralBank')['n_words'].agg(['count','mean','std']).round(0))

# Est-ce que les textes très longs disparaissent ?
print(f"\nTextes > 10000 mots avant filtre : {(df_topic['n_words'] > 10000).sum()}")
print(f"Textes > 10000 mots après filtre : {(df_filtered['n_words'] > 10000).sum()}")